In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import numpy as np
import torch
import pprint

# add parent dir so importing top level files works in notebook subdir
parent_dir = str(Path().resolve().parent)
sys.path.insert(0, parent_dir)

from llm import qwen_utils, tools
from autoencoder.model_qwen import QwenAutoencoder

/home/guests/nicolas_stellwag/surgery-scene-graphs/.pixi/envs/default/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [23]:
graph_dir = Path("output/qwen3_da3_flow/video12_19500/graph")
ae_path = Path("data/preprocessed/qwen3_da3_subsampled/video12_19500/autoencoder_me/best_ckpt.pth")

print(f"Graph dir: {graph_dir}")
print(f"Autoencoder path: {ae_path}")

Graph dir: output/qwen3_da3_flow/video12_19500/graph
Autoencoder path: data/preprocessed/qwen3_da3_subsampled/video12_19500/autoencoder_me/best_ckpt.pth


In [24]:
# Load autoencoder
autoencoder = QwenAutoencoder(input_dim=4096 * 4, latent_dim=3).to("cuda")
autoencoder.load_state_dict(torch.load(ae_path, map_location="cuda"))
autoencoder.eval()
print(f"Autoencoder loaded from {ae_path}")

# Load graph data and create toolkit (accepts npz directly)
toolkit = tools.GraphTools(
    positions=np.load(graph_dir / "positions.npy"),
    clusters=np.load(graph_dir / "clusters.npy"),
    centroids=np.load(graph_dir / "c_centroids.npy"),
    centers=np.load(graph_dir / "c_centers.npy"),
    extents=np.load(graph_dir / "c_extents.npy"),
    adjacency=np.load(graph_dir / "graph.npy"),
    bhattacharyya_coeffs=np.load(graph_dir / "bhattacharyya_coeffs.npy"),
    qwen_feats=np.load(graph_dir / "c_qwen_feats.npz"),
    patch_latents_through_time=np.load(graph_dir / "patch_latents_through_time.npy"),
    autoencoder=autoencoder,
)
graph_tools = toolkit.get_all_tools()
print(f"Loaded {len(graph_tools)} tools: {list(graph_tools.keys())}")
print(f"Graph has {len(np.unique(toolkit.clusters))} nodes, {toolkit.adjacency.shape[0]} timesteps")

Autoencoder loaded from data/preprocessed/qwen3_da3_subsampled/video12_19500/autoencoder_me/best_ckpt.pth
Loaded 7 tools: ['node_distances_through_time', 'node_overlap_scores_through_time', 'node_overlap_position_at_time', 'inspect_highres_node_at_time', 'inspect_node_through_time', 'inspect_scene_at_time', 'voxelize_scene']
Graph has 10 nodes, 20 timesteps


In [6]:
# Load patched Qwen3-VL model with custom feature injection support
model, processor = qwen_utils.get_patched_qwen3()
print(f"Model loaded on device: {model.device}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded on device: cuda:0


# Raw Features

In [39]:
lhook_feats = toolkit.qwen_feats["2"][10] # should be l hook
liver_feats = toolkit.qwen_feats["8"][10] # should be liver
gallbladder_feats = toolkit.qwen_feats["5"][10] # should be gallbladder

messages = [
    {"role": "system", "content": [{"type": "text", "text": "Your are a helpful assistant."}]},
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Image 1:"},
            {"type": "image", "image": None},
            {"type": "text", "text": "Image 2:"},
            {"type": "image", "image": None},
            {"type": "text", "text": "Image 3:"},
            {"type": "image", "image": None},
            {"type": "text", "text": "\n\nWhat do you see in each image?"},
        ],
    },
]

lhook_feats = torch.tensor(lhook_feats, device=model.device).float()[:1]
liver_feats = torch.tensor(liver_feats, device=model.device).float()[:1]
gallbladder_feats = torch.tensor(gallbladder_feats, device=model.device).float()[:1]

qwen_utils.generate_with_vision_features(
    messages=messages,
    vision_features=[
        lhook_feats,
        liver_feats,
        gallbladder_feats,
    ],
    model=model,
    processor=processor,
    qwen_version="qwen3",
)


"Got it, let's look at each image one by one. First, Image 1: there's a syringe or some kind of needle being inserted into a dark surface, maybe a piece of tissue or a medical sample. The needle is metallic, and the background is dark. Image 2: it shows a close-up of what looks like a tissue or maybe a skin sample with some texture, possibly a cut surface or a biopsy. The color is a bit grayish, with some uneven areas. Image 3: similar to Image 2, but maybe a different section or angle. It's also a grayish tissue sample, with visible textures and maybe some irregularities. I need to describe each image clearly.\n</think>\n\nHere’s a description of what’s visible in each image:  \n\n### **Image 1**  \nA close - up of a **metallic needle or syringe tip** (likely medical) inserted into a **dark, textured surface** (possibly a biological tissue sample, like a skin or organ section). The needle is prominent, with a sharp, cylindrical shape, and the background appears blurred or dark, emphas

# Spatial

In [37]:
spatial_tool_names = [
    "node_overlap_position_at_time",
    "inspect_highres_node_at_time",
    "voxelize_scene",
]

spatial_tool_call_limits = {
    "inspect_highres_node_at_time": 5,
    "voxelize_scene": 5,
}

spatial_tool_dict = toolkit.get_tools_by_name(spatial_tool_names)

# spatial_system_prompt = """
# # Task
# You are an expert visceral surgeon locating objects or actions in a cholecystectomy scene.
# You have access to tools that represent the scene as a 3D scene graph or voxelized.
# Your initial scene representation is a set of graph nodes at .

# # Scene representation
# - Coordinate system:
#     - increase in x means movement to the right
#     - increase in y means movement to the bottom
#     - increase in z means movement away from the camera

# # Rules
# - Strictly respect tool call limits. (Initially given by user prompt, updated by tool responses.)
# - Reason very briefly in each turn and optimize for answer speed.

# # Answer format
# Your final answer should always contain exactly one 3D point as a JSON object. Never refuse!
# Report the point in the graph's 3D coordinate system.
# """

# spatial_prompt = """
# Identify the 3D location of 'grasper' in the scene.

# Example output (format only): ... {"x": X, "y": Y, "z": Z} ...
# """

spatial_system_prompt = """
# Task
You are an expert visceral surgeon locating objects or actions in a cholecystectomy scene.
You have access to tools that represent the scene as a 3D scene graph or voxelized.
Your initial scene representation is a set of graph nodes at the timestep you should use for localization.

# Scene representation
- Coordinate system:
    - increase in x means movement to the right
    - increase in y means movement to the bottom
    - increase in z means movement away from the camera

# Rules
- Strictly respect tool call limits. (Initially given by user prompt, updated by tool responses.)
- Reason very briefly in each turn and optimize for answer speed.
"""
spatial_prompt = """
Please describe what you see in all initial nodes' rough images. (Don't call any tools.)
"""

timestep_idx = 4

In [38]:
Path("spatial.rrd").unlink(missing_ok=True)
toolkit.start_recording("spatial.rrd")
response = qwen_utils.prompt_graph_agent(
    question=spatial_prompt,
    node_feats=toolkit.qwen_feats,
    initial_timestep_idx=timestep_idx,
    node_centers=toolkit.point_o2n(toolkit.centers),
    node_centroids=toolkit.point_o2n(toolkit.centroids),
    node_extents=toolkit.distance_o2n(toolkit.extents),
    model=model,
    processor=processor,
    qwen_version="qwen3",
    system_prompt=spatial_system_prompt,
    tools=spatial_tool_dict,
    tool_call_limits=spatial_tool_call_limits,
    verbose=True,
)


[15:59:06] --- Iteration 0 ---
[15:59:41] [Assistant Response]:
Okay, I need to describe what's in all the initial nodes' rough images for timestep 4. Let's look at each node's rough_image.

Node 0: The image shows a close-up of tissue with a metallic instrument, probably a surgical tool, on the left. There's a yellowish area, maybe fat or connective tissue, and some fluid or blood. The scene is inside the body, likely the abdomen.

Node 1: This has a metallic tube or cannula visible, possibly an endoscope. The background is dark with some tissue, maybe the gastrointestinal tract.

Node 2: Another tube, maybe a scope, with a light source. The image is a bit blurry, but there's a white tip, possibly a scope tip. Tissue is visible around it.

Node 3: Similar to Node 0, tissue with a metallic instrument. The tissue looks like it's part of the gallbladder area, maybe the cystic duct.

Node 4: There's a tube or scope in the upper part, and tissue with some fluid or blood. The yellowish are

In [35]:
pprint.pprint(response["message_history"])

[{'content': [{'text': '\n'
                       '# Task\n'
                       'You are an expert visceral surgeon locating objects or '
                       'actions in a cholecystectomy scene.\n'
                       'You have access to tools that represent the scene as a '
                       '3D scene graph or voxelized.\n'
                       'Your initial scene representation is a set of graph '
                       'nodes at timestep 0.\n'
                       '\n'
                       '# Scene representation\n'
                       '- Coordinate system:\n'
                       '    - increase in x means movement to the right\n'
                       '    - increase in y means movement to the bottom\n'
                       '    - increase in z means movement away from the '
                       'camera\n'
                       '\n'
                       '# Rules\n'
                       '- Strictly respect tool call limits. (Initially given '
  

In [13]:
toolkit.log_final_prediction(
    position=toolkit.point_n2o(np.array([-1.4969, 0.1585, 1.2104])),
    timestep_idx=timestep_idx,
    label="grasper grasping gallbladder",
    entity_name="zz_final_prediction",
)
toolkit.stop_recording()

# Temporal

In [32]:
tool_names = [
    "node_distances_through_time",
    "node_overlap_scores_through_time",
    "inspect_node_through_time",
    "inspect_scene_at_time",
    "inspect_highres_node_at_time",
    "voxelize_scene",
]

tool_call_limits = {
    "inspect_node_through_time": 5,
    "inspect_scene_at_time": 3,
    "inspect_highres_node_at_time": 5,
    "voxelize_scene": 5,
}

tool_dict = toolkit.get_tools_by_name(tool_names)

# tool_dict = toolkit.get_tools_by_name(["voxelize_scene"])
# tool_call_limits = {"voxelize_scene": 5}

In [ ]:
system_prompt = """
# Task
You are an expert visceral surgeon analyzing a cholecystectomy scene for temporal reasoning.
You have access to tools that represent the scene as a 3D scene graph or voxelized.
Your initial scene representation is a set of graph nodes at timestep 0.

# Scene representation
- Coordinate system:
    - increase in x means movement to the right
    - increase in y means movement to the bottom
    - increase in z means movement away from the camera

# Rules
- Strictly respect tool call limits. (Initially given by user prompt, updated by tool responses.)
- Reason very briefly in each turn and optimize for answer speed.

# Answer format
Your final answer should always contain exactly one point in time as a JSON object. Never refuse!
Report the point in time in seconds.
"""

prompt = """
Identify a point in time to answer the question: 'When does the grasper start grasping the gallbladder?'.

The graph has 20 timesteps (t=0 to t=19).

Example output (format only): ... {"timestep": X} ...
"""

# system_prompt = "You are an expert visceral surgeon analyzing a cholecystectomy scene for temporal reasoning. We are currently testing your capabilities to interact with the scene graph. Reason briefly, optimize for answer speed."

# prompt = "Do NOT call any tools, just tell be a joke. I want to test if inference speed is slow or if the tool too is slow."

In [34]:
# Clear CUDA cache before generation to avoid OOM
torch.cuda.empty_cache()

toolkit.start_recording("temporal.rrd")
response = qwen_utils.prompt_graph_agent(
    question=prompt,
    node_feats=toolkit.qwen_feats,
    initial_timestep_idx=0,
    node_centers=toolkit.point_o2n(toolkit.centers),
    node_centroids=toolkit.point_o2n(toolkit.centroids),
    node_extents=toolkit.distance_o2n(toolkit.extents),
    model=model,
    processor=processor,
    qwen_version="qwen3",
    system_prompt=system_prompt,
    tools=tool_dict,
    tool_call_limits=tool_call_limits,
    verbose=True,
)
toolkit.stop_recording()
# if type(response) == dict:
#     pprint.pprint(response["message_history"])
# else:
#     print(response)


[13:10:13] --- Iteration 0 ---
[13:11:34] [Assistant Response]:
Okay, I need to figure out when the grasper starts grasping the gallbladder in this cholecystectomy scene. Let's see. The user provided a graph with nodes at timestep 0, and there are 20 timesteps total from 0 to 19.

First, I should identify which nodes represent the grasper and the gallbladder. The nodes have lowres-visual-descriptor images. The grasper is probably a surgical instrument, so looking at the descriptors: nodes 1, 3, 7, 8, 10 might be instruments. Wait, node 1 has a "clamp" or "grasper" look in its descriptor image. Similarly, node 3 has a similar instrument but maybe different. Wait, node 1's descriptor is "a pair of clamps" or something, maybe a grasper. The gallbladder is likely a structure with a textured appearance, maybe the nodes with more organic structures, like node 2, 4, 5, 8, etc.

Wait, node 2's descriptor is "a fleshy structure with visible blood vessels," which could be the gallbladder. Node 

# Check tool call recording

In [22]:
# Test tool call logging with fixed order

print("Starting tool call logging test...")
print(f"Graph: {len(np.unique(toolkit.clusters))} nodes, {toolkit.adjacency.shape[0]} timesteps")

# Start recording
toolkit.start_recording("test_tool_calls.rrd")
print("Recording started, initial graph logged")

# Get all tools
all_tools = toolkit.get_all_tools()
print(f"Available tools: {list(all_tools.keys())}")

# Call each tool exactly once in a fixed order
print("\n--- Calling tools in order ---")

# 1. node_distances_through_time
print("1. Calling node_distances_through_time(1, 3)...")
result = all_tools["node_distances_through_time"][0](node_id_1=1, node_id_2=3)
print(f"   Counter: {toolkit.call_counter}")

# 2. node_overlap_scores_through_time  
print("2. Calling node_overlap_scores_through_time(1, 3)...")
result = all_tools["node_overlap_scores_through_time"][0](node_id_1=1, node_id_2=3)
print(f"   Counter: {toolkit.call_counter}")

# 3. node_overlap_position_at_time
print("3. Calling node_overlap_position_at_time(1, 3, 5)...")
result = all_tools["node_overlap_position_at_time"][0](node_id_1=1, node_id_2=3, timestep=5)
print(f"   Counter: {toolkit.call_counter}")

# 4. inspect_highres_node_at_time
print("4. Calling inspect_highres_node_at_time(2, 10)...")
result = all_tools["inspect_highres_node_at_time"][0](node_id=2, timestep=10)
print(f"   Counter: {toolkit.call_counter}")

# 5. inspect_node_through_time
print("5. Calling inspect_node_through_time(2)...")
result = all_tools["inspect_node_through_time"][0](node_id=2)
print(f"   Counter: {toolkit.call_counter}")

# 6. inspect_scene_at_time
print("6. Calling inspect_scene_at_time(10)...")
result = all_tools["inspect_scene_at_time"][0](timestep=10)
print(f"   Counter: {toolkit.call_counter}")

# 7. voxelize_scene
print("7. Calling voxelize_scene(5)...")
result = all_tools["voxelize_scene"][0](timestep=5)
print(f"   Counter: {toolkit.call_counter}")

# Stop recording
toolkit.stop_recording()

Starting tool call logging test...
Graph: 14 nodes, 20 timesteps
Recording started, initial graph logged
Available tools: ['node_distances_through_time', 'node_overlap_scores_through_time', 'node_overlap_position_at_time', 'inspect_highres_node_at_time', 'inspect_node_through_time', 'inspect_scene_at_time', 'voxelize_scene']

--- Calling tools in order ---
1. Calling node_distances_through_time(1, 3)...
   Counter: 1
2. Calling node_overlap_scores_through_time(1, 3)...
   Counter: 2
3. Calling node_overlap_position_at_time(1, 3, 5)...
   Counter: 3
4. Calling inspect_highres_node_at_time(2, 10)...
   Counter: 4
5. Calling inspect_node_through_time(2)...
   Counter: 5
6. Calling inspect_scene_at_time(10)...
   Counter: 6
7. Calling voxelize_scene(5)...
   Counter: 7
